In [1]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### deconv module

In [2]:
upconv1 = nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1)
bn1 = nn.BatchNorm2d(256)
relu1 = nn.ReLU()
upconv2 = nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1)
bn2 = nn.BatchNorm2d(256)
relu2 = nn.ReLU()
upconv3 = nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1)
bn3 = nn.BatchNorm2d(256)
relu3 = nn.ReLU()

In [3]:
class DeconvLayer(nn.Module):
    def __init__(self, num_deconv_layers):
        super(DeconvLayer, self).__init__()
        layers = []
        for _ in range(num_deconv_layers):
            layers.append(nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1))
            layers.append(nn.BatchNorm2d(256))
            layers.append(nn.ReLU())
        self.deconv_layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.deconv_layers(x)

upconv = DeconvLayer(3)

In [4]:
final_layer = nn.Conv2d(256, 17, kernel_size=1, padding=0)

In [5]:
class PoseEstimationModel(nn.Module):
    def __init__(self):
        super(PoseEstimationModel, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-2])
        self.reduce_channels = nn.Conv2d(2048, 256, kernel_size=1)
        self.upconv = DeconvLayer(3)
        self.final_layer = nn.Conv2d(256, 17, kernel_size=1, padding=0)

    def forward(self, x):
        x = self.resnet(x)
        x = self.reduce_channels(x)
        x = self.upconv(x)
        x = self.final_layer(x)
        return x

model = PoseEstimationModel()
print(model)

/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/jovyan/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]


PoseEstimationModel(
  (resnet): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): 

In [6]:
%%time
np_input = np.zeros((1, 3, 256, 192), dtype=np.float32)
torch_input = torch.tensor(np_input, dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_input = torch_input.to(device)
model = model.to(device)
print('input shape')
print(torch_input.shape)
print('\n')

with torch.no_grad():
    torch_output = model(torch_input)

print('output shape')
print(torch_output.shape)
print(torch_output[0, :10, :10, :10])

input shape
torch.Size([1, 3, 256, 192])


output shape
torch.Size([1, 17, 64, 48])
tensor([[[ 5.8984e-02,  1.1033e-01,  3.4904e-01, -1.3893e-01,  1.7058e-01,
          -2.2425e-02,  3.3275e-01, -2.3589e-02,  1.3443e-01, -3.8962e-02],
         [ 2.1049e-01,  3.3480e-01,  2.2409e-01,  3.0779e-01,  5.5541e-01,
           2.8624e-01,  1.9153e-01,  3.7565e-01,  2.6710e-01,  2.1069e-02],
         [-1.6097e-01, -2.3674e-01, -1.9612e-01, -2.1954e-01,  5.2145e-02,
           2.1265e-01,  4.0306e-01,  3.1835e-02,  3.8465e-01,  5.1818e-01],
         [ 1.7487e-01,  3.5345e-01,  1.8782e-01,  8.0457e-01,  2.6242e-01,
           5.0029e-01,  2.8229e-01,  1.7271e-01,  2.7138e-01,  6.2116e-01],
         [-8.1349e-02,  2.9339e-01,  2.3560e-01,  4.0732e-02, -8.1834e-02,
           2.7640e-01, -3.1833e-01, -2.1953e-01,  4.4471e-01,  2.1120e-01],
         [ 3.4543e-01,  6.4787e-01,  5.8125e-01,  2.8799e-01,  7.5328e-01,
           1.8833e-01,  2.4572e-01,  9.5872e-01,  3.4948e-01,  1.5067e-01],
         [